# Learning path

This page is an optional course with exercises in the Advanced group. It is for
readers who have completed the [Quickstart](quickstart.ipynb) and
[Five modelling decisions](five_decisions.ipynb) and want to work through the
methodology; readers who only need to fit a model can continue with
[glmboost](glmboost.ipynb) instead. The course is organized around the decisions a
statistical analysis must make. After completing it, a reader can identify the quantity a boosting model predicts,
reconstruct one component-wise update, choose a validation design, diagnose a
distributional failure, and separate predictive evidence from a causal claim.
The exercises use algebra from an introductory regression course. The linked
methodology chapters supply the derivations needed for each solution.

A first pass takes about four hours: read the short explanation at each stage, run
the code, write an answer before opening the solution, and keep the final assignment
for a separate session. Use the API to calculate each quantity, then explain its statistical meaning.


## Stage 1: loss and prediction target

A response column does not determine what its fitted value means. Squared error
targets a conditional mean, absolute error targets a conditional median, and an
asymmetric quantile loss targets the value below which a specified fraction of
outcomes falls. The target changes while the response and predictor remain fixed.
The [five decisions](five_decisions.ipynb) page maps common scientific targets to
families; [families and losses](../methodology/families.ipynb) derives the gradients.

**Check.** A hospital needs both expected bed-days for capacity planning and a bed-day
value exceeded by only one patient in ten. Name the two targets and explain why one
Gaussian mean fit cannot represent both.

<details><summary>Worked solution</summary>

Expected bed-days is the conditional mean. The planning threshold is the conditional
90th percentile, fitted with `Quantile(tau=0.9)`. A mean describes the first moment; a
90th percentile describes a tail location. They coincide only under additional
distributional restrictions and even then are not the same quantity.

</details>


## Stage 2: component-wise updates

For squared error, the negative gradient is the current residual. Each base learner
fits that residual, and the learner with the smallest residual sum of squares supplies
the next update. The learning rate multiplies the update; it does not change which
learner is selected at that iteration. The [worked boosting example](../methodology/worked_example.ipynb)
continues this calculation through a complete path.


In [1]:
import numpy as np
import polars as pl

x1 = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
x2 = np.array([1.0, -1.0, 1.0, -1.0, 1.0])
y = np.array([-2.2, -0.7, 0.4, 1.3, 2.7])
offset = y.mean()
negative_gradient = y - offset
slopes = np.array([
    np.dot(x1, negative_gradient) / np.dot(x1, x1),
    np.dot(x2, negative_gradient) / np.dot(x2, x2),
])
candidate_rss = np.array([
    np.sum((negative_gradient - slopes[0] * x1) ** 2),
    np.sum((negative_gradient - slopes[1] * x2) ** 2),
])
selected = int(np.argmin(candidate_rss))
learning_rate = 0.1
first_fit = offset + learning_rate * slopes[selected] * [x1, x2][selected]
pl.DataFrame({
    "learner": ["x1", "x2"],
    "slope_on_negative_gradient": slopes,
    "candidate_rss": candidate_rss,
    "selected": [selected == 0, selected == 1],
})


learner,slope_on_negative_gradient,candidate_rss,selected
str,f64,f64,bool
"""x1""",1.18,0.096,true
"""x2""",8.8818e-17,14.02,false


**Check.** Recompute `first_fit` with learning rate 0.5. State what changes and what
does not change in this first iteration.

<details><summary>Worked solution</summary>

The selected learner, its fitted slope, and both candidate residual sums of squares do
not change because selection occurs before shrinkage. The displacement from the offset
is five times as large. Later selections can change because the larger first update
leaves a different residual for the second iteration.

</details>


## Stage 3: learner flexibility and selection

A linear learner estimates one slope after accounting for its intercept convention. A
penalized spline estimates several basis coefficients, while its penalty discourages
rapid changes. Degrees of freedom calibrate the flexibility offered to each learner;
they are not the number of columns in the spline design. Unequal flexibility can make
selection favor the learner that can fit more residual patterns. See
[base learners](base_learners.ipynb) and [penalties and degrees of freedom](../methodology/penalties_df.ipynb).

**Check.** Two predictors contain the same signal. One enters as `bols(x)` and the other
as an almost unpenalized spline. Explain why selection frequency does not measure which
predictor is the true cause.

<details><summary>Worked solution</summary>

The spline can fit a larger class of residual shapes, so it may win the numerical
competition even when the predictors carry equivalent information. Correlation also
lets substitute terms divide selections across resamples. Selection frequency records
an algorithm under a learner menu; causality additionally requires an intervention and
identification assumptions.

</details>


## Stage 4: validation design

Cross-validation estimates performance for the split and score it implements. Random
row folds approximate exchangeable future rows. Grouped folds keep repeated records
from the same sampling unit together. Rolling folds train on the past and judge a fixed
future horizon. Tuning belongs inside the training part of any final evaluation split.
The [model-selection guide](model_selection.ipynb) covers stopping, and
[real-data validation](real_data.ipynb) covers grouped and ordered designs.

**Check.** A dataset contains monthly records for each shop, and the deployment task is
to forecast the next quarter for shops already observed. Compare random row, grouped,
and rolling folds.

<details><summary>Worked solution</summary>

Random rows train on later months and leak each shop's neighboring records. Grouped
folds estimate performance on entirely new shops, a harder target than deployment. A
rolling origin with all prior shops in training and the next quarter as the judge set
matches the stated forecast. If performance for new shops also matters, it requires a
second design that groups shops and respects time.

</details>


## Stage 5: distributional models

A location model can fit the conditional mean while missing how uncertainty changes.
An LSS model assigns a separate additive predictor to each distribution parameter. A
parameter named `mu` is not automatically the response mean: the family definition
determines its interpretation. Randomized PIT values should be uniform when the full
conditional distribution is calibrated, including jumps for discrete or mixed data.
See [distributional boosting](gamboostlss.ipynb) and
[diagnostics](diagnostics.ipynb).

**Check.** A zero-adjusted Gamma model has zero probability `nu` and positive-component
mean `mu`. Give the unconditional response mean and explain why comparing binned
outcomes directly with `mu` misdiagnoses mean calibration.

<details><summary>Worked solution</summary>

The response mean is `(1-nu)*mu`: zero outcomes contribute no positive amount, and the
positive component occurs with probability `1-nu`. Comparing observations with `mu`
ignores the atom and treats the component mean as the mixture mean.

</details>


## Stage 6: effect bands, prediction intervals and selection frequencies

A pointwise effect band describes uncertainty in a fitted function at one predictor
value. A simultaneous band covers the complete function under its stated procedure. A
prediction interval targets a future outcome, so it also contains irreducible response
variation. A stability-selection frequency describes repeated selection under a
resampling algorithm and is not a coefficient confidence level. The
[inference guide](inference.ipynb), [conformal chapter](../methodology/conformal.ipynb),
and [stability-selection guide](stability_selection.ipynb) state their assumptions.

**Check.** A 95% pointwise band excludes zero over most of a smooth effect. State two
claims that do not follow.

<details><summary>Worked solution</summary>

The band does not imply 95% simultaneous coverage of the whole curve. It also does not
identify a causal effect: learner selection, specification choices, confounding, and
post-treatment adjustment are outside the pointwise sampling calculation.

</details>


## Stage 7: causal estimands

A treatment-effect estimand compares potential outcomes under two treatment settings
for a named population. Prediction alone supplies neither the missing potential outcome
nor the assumptions that identify its average. `CausalSpec` records timing, adjustment,
and consistency, exchangeability, positivity, and interference assumptions before
`aipw` estimates an effect. The [causal workflow](causal.ipynb) includes overlap, balance,
sensitivity, and negative-control checks.

**Check.** A variable measured after treatment predicts the outcome and improves held-out
error. Explain why that is insufficient reason to adjust for it.

<details><summary>Worked solution</summary>

The variable may be a mediator through which treatment changes the outcome or a collider
affected by treatment and another outcome cause. Conditioning on it can remove part of
the total effect or open a noncausal path. Its predictive value does not determine its
position in the causal graph; timing and subject-matter structure do.

</details>


## Final assignment: a complete analysis

Choose a response and at least three baseline predictors from a documented dataset.
Submit a short notebook containing the following artifacts:

1. Define the prediction target, response support, observation unit, and deployment
   population. State whether the claim is descriptive, predictive, or causal.
2. Fit one linear specification and one plausible nonlinear alternative. Keep the loss,
   eligible sample, folds, score, and tuning budget fixed for their comparison.
3. Reconstruct one boosting update or one fold score from retained arrays. Label every
   quantity by scale and denominator.
4. Diagnose residual shape and dependence. Revise one specification in response to a
   diagnostic, then report whether held-out performance changes.
5. Report uncertainty suited to the prediction task. Add a counterexample or alternative
   specification capable of overturning the main interpretation.
6. If making a causal claim, add the intervention, time zero, adjustment rationale,
   overlap and balance evidence, and a sensitivity or negative-control analysis. If those
   conditions cannot be defended, retain the predictive claim.

A complete submission lets another reader reproduce the eligible rows, folds, fitted
target, score, and conclusion. Match the interpretation to that evidence. Define the
target and use folds that hold out the prediction unit; leakage or an undefined target
invalidates a lower held-out score.
